> Inherited v0.4 reference notebook. Begin with `00_start_here.ipynb` and the v0.5 README. This notebook does not implement the matched-feature real-data qualification or establish current scientific validity.

# 02. PE label and prior audit

Start from the official lightweight summary tables, then compare them with the deep PESummary inspection reports. This notebook discovers candidates; it does not freeze a production label or reconstruct a prior.

In [1]:
from pathlib import Path
import json
import os
import pandas as pd

inventory_dir = Path(os.environ["GWTC_RESULTS_ROOT"]) / "pilot" / "inventory"
index_path = inventory_dir / "pe_release_index.json"
index = json.loads(index_path.read_text())
index["ready_for_label_audit"], index_path

(True,
 PosixPath('/Projects/Data/macias/924994587/GW_Project_Data/results/pilot/inventory/pe_release_index.json'))

In [2]:
candidate_rows = []
for release, payload in index["releases"].items():
    if not payload.get("present", False):
        continue
    for event_id, event in payload["events"].items():
        for row in event["rows"]:
            candidate_rows.append({"release": release, "event_id": event_id, **row})
candidates = pd.DataFrame(candidate_rows)
candidates[[
    "release", "event_id", "result_samples_key", "result_file_name",
    "mass_1_source_median", "mass_2_source_median", "redshift_median"
]] if len(candidates) else candidates

,release,event_id,result_samples_key,result_file_name,mass_1_source_median,mass_2_source_median,redshift_median
0,o4a,GW230627_015337,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.483478,5.714881,0.066302
1,o4a,GW230627_015337,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.449413,5.742252,0.065766
2,o4a,GW230627_015337,C00:SEOBNRv5PHM,IGWN-GWTC4p1-18965dda8_5-GW230627_015337-combi...,8.422318,5.762235,0.065132
3,o4a,GW230726_002940,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.859585,27.815026,0.373798
4,o4a,GW230726_002940,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.554629,27.895568,0.372801
5,o4a,GW230726_002940,C00:NRSur7dq4,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.427186,27.972493,0.372840
6,o4a,GW230726_002940,C00:SEOBNRv5PHM,IGWN-GWTC4p1-18965dda8_5-GW230726_002940-combi...,35.365654,27.940824,0.371909
7,o4a,GW230814_230901,C00:IMRPhenomXO4a,IGWN-GWTC4p1-18965dda8_5-GW230814_230901-combi...,33.242359,28.295266,0.059460
8,o4a,GW230814_230901,C00:IMRPhenomXPHM-SpinTaylor,IGWN-GWTC4p1-18965dda8_5-GW230814_230901-combi...,33.753063,28.286814,0.060964
9,o4a,GW230814_230901,C00:Mixed,IGWN-GWTC4p1-18965dda8_5-GW230814_230901-combi...,33.740290,28.196569,0.062482


In [3]:
deep_rows = []
for path in sorted(inventory_dir.glob("pe_*_deep.json")):
    report = json.loads(path.read_text())
    event_id = path.name.split("_")[1] if path.name.startswith("pe_GW") else path.stem
    for analysis in report.get("analyses", []):
        deep_rows.append({
            "report": path.name,
            "event_id": event_id,
            "label": analysis["label"],
            "sample_count": analysis["sample_count"],
            "parameter_count": len(analysis["parameters"]),
            "has_mass_1_source": "mass_1_source" in analysis["parameters"],
            "has_mass_2_source": "mass_2_source" in analysis["parameters"],
            "has_redshift": "redshift" in analysis["parameters"],
        })
pd.DataFrame(deep_rows)

,report,event_id,label,sample_count,parameter_count,has_mass_1_source,has_mass_2_source,has_redshift
0,pe_GW230627_015337_deep.json,GW230627,C00:IMRPhenomXPHM-SpinTaylor,16191,149,True,True,True
1,pe_GW230627_015337_deep.json,GW230627,C00:Mixed,32382,149,True,True,True
2,pe_GW230627_015337_deep.json,GW230627,C00:SEOBNRv5PHM,16577,149,True,True,True


## Required decision record

For each event, record: (1) every released candidate label, (2) the official population-analysis label/mixing policy, (3) available prior and configuration metadata, (4) sampled coordinates, (5) source-frame transformations and Jacobians, and (6) an independent numerical prior check. Only then update `config/pe_mapping.toml`. A tutorial waveform-priority example is not automatically the population-analysis policy.